<a href="https://www.kaggle.com/code/smeyra/llm-kurumsal-dok-man-y-nlendirme-ve-bilgi-getirme?scriptVersionId=315019661" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Proje: Kurumsal Doküman Yönlendirme ve Bilgi Getirme Sistemi

**Amaç:** Bu projede, bir şirketin iç dokümanları arasında akıllı bir arama sistemi geliştireceğiz. Sistem, kullanıcının sorduğu sorunun önce hangi departmanla ilgili olduğunu (İK, Muhasebe, Satın Alma vb.) **sınıflandıracak**, ardından sadece o departmanın dokümanları içinde anlamsal bir arama yaparak en ilgili cevabı **bulacaktır**.

**Senaryo:** Bir şirket çalışanının, "Yeni bir dizüstü bilgisayarı nasıl talep edebilirim?" diye sorduğunu düşünün. Sistemimiz bu sorunun tüm şirket dokümanlarında aranması yerine, önce "Satın Alma" veya "IT" kategorisine ait olduğunu tespit etmeli, sonra bu kategoriye ait dokümanlar içinden en uygun olanını bulmalıdır. Bu iki aşamalı yaklaşım (Sınıflandır ve Getir - Classify & Retrieve), RAG sistemlerinin temelini oluşturur.

## Adım 1: Kurulum ve Veri Setinin Hazırlanması

Gerekli kütüphaneleri yükleyerek işe başlayalım ve farklı departmanlara ait örnek dokümanlardan oluşan kurumsal bilgi havuzumuzu (corpus) oluşturalım.

In [1]:
# Gerekli kütüphaneleri yüklemek için (eğer yüklü değilse):
# !pip install scikit-learn sentence-transformers torch transformers numpy

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# 5 farklı departmana ait örnek dokümanlar
corpus = {
    "İnsan Kaynakları": [
        "Yıllık izin talepleri, İK portalı üzerinden en az bir hafta önceden yapılmalıdır.",
        "Performans değerlendirme süreci her yılın Haziran ve Aralık aylarında gerçekleştirilir.",
        "Şirketimiz çalışanlarına özel sağlık sigortası sağlamaktadır.",
        "Yeni işe başlayanlar için oryantasyon programı ilk hafta zorunludur."
    ],
    "Muhasebe": [
        "Maaş ödemeleri her ayın son iş günü banka hesaplarına yatırılır.",
        "Masraf beyan formları, harcamanın ardından en geç 5 iş günü içinde teslim edilmelidir.",
        "Avans talepleri için Muhasebe departmanından onay alınması gerekmektedir.",
        "Tüm faturaların dijital kopyaları muhasebe sistemine yüklenmelidir."
    ],
    "Satın Alma": [
        "Ofis malzemeleri ve kırtasiye ihtiyaçları için talep formu doldurulmalıdır.",
        "Yeni bir dizüstü bilgisayar veya monitör talebi, IT departmanının onayıyla Satın Alma birimine iletilir.",
        "Tedarikçi seçim sürecinde en az üç farklı firmadan teklif alınır.",
        "Onaylanmış satın alma siparişleri ay sonunda raporlanır."
    ],
    "IT Destek": [
        "Parola sıfırlama işlemleri için self-servis IT portalını kullanabilirsiniz.",
        "VPN bağlantı sorunları için lütfen IT destek ekibine bir destek bileti oluşturun.",
        "Yazılım kurulum talepleri, güvenlik onayı alındıktan sonra gerçekleştirilir.",
        "Bilgisayarınızda donanımsal bir arıza olduğunu düşünüyorsanız, cihazı IT departmanına getiriniz."
    ],
    "İdari İşler": [
        "Şirket içi yemekhane hizmeti hafta içi her gün 12:00-14:00 saatleri arasındadır.",
        "Personel servis güzergahları ve saatleri intranet sayfasında yayınlanmaktadır.",
        "Ofis temizliği ve düzeni ile ilgili talepler İdari İşler birimine bildirilmelidir.",
        "Misafirler için otopark kullanımı, bir gün önceden resepsiyona bildirilmelidir."
    ]
}

print("Kurumsal bilgi havuzu ve departmanlar oluşturuldu.")

2026-04-28 10:04:23.121325: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777370663.330780      12 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777370663.391143      12 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Kurumsal bilgi havuzu ve departmanlar oluşturuldu.


## Adım 2: Doküman Sınıflandırıcı (Router) Geliştirme

Bu adımda, gelen bir sorgunun hangi departmanla ilgili olduğunu tahmin edecek bir sınıflandırıcı model eğiteceğiz. Bunun için önce tüm dokümanlarımızı vektörlere dönüştürmemiz gerekiyor.

In [2]:
# Sınıflandırıcıyı eğitmek için veri ve etiketleri hazırlayalım
documents = []
labels = []
for department, docs in corpus.items():
    documents.extend(docs)
    labels.extend([department] * len(docs))

# Modelin yüklenmesi
sbert_model = SentenceTransformer('dbmdz/bert-base-turkish-cased')

def train_classifier(docs, labels):
    """Dokümanları vektörleştirir ve bir sınıflandırıcı eğitir."""
    # TODO: Sentence-BERT kullanarak tüm dokümanları vektörlere dönüştürün (embedding).
    # sbert_model.encode() metodunu kullanın.
    doc_embeddings = sbert_model.encode (docs)
    
    # TODO: Veri setini eğitim ve test olarak ayırın.
    # train_test_split fonksiyonunu kullanın. test_size=0.2, random_state=42 iyi bir başlangıçtır.
    X_train, X_test, y_train, y_test =  train_test_split(doc_embeddings, labels, test_size=0.2, random_state=42)


    # TODO: Lojistik Regresyon modelini oluşturun ve eğitin.
    # model.fit() metodunu kullanın.
    
    classifier = LogisticRegression()
    classifier.fit(X_train, y_train)
    
    # Modelin performansını test seti üzerinde değerlendirelim
    print("Sınıflandırıcı Modeli Performansı:")
    predictions = classifier.predict(X_test)
    print(classification_report(y_test, predictions))
    
    return classifier

# Sınıflandırıcıyı eğitelim
document_classifier = train_classifier(documents, labels)

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Sınıflandırıcı Modeli Performansı:
                  precision    recall  f1-score   support

       IT Destek       0.50      1.00      0.67         1
        Muhasebe       0.00      0.00      0.00         0
      Satın Alma       0.00      0.00      0.00         0
     İdari İşler       0.00      0.00      0.00         1
İnsan Kaynakları       0.00      0.00      0.00         2

        accuracy                           0.25         4
       macro avg       0.10      0.20      0.13         4
    weighted avg       0.12      0.25      0.17         4



/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.

## Adım 3: Bilgi Getirme (Retriever) Fonksiyonu

Bu fonksiyon, bir sorgu ve **sadece belirli bir departmana ait doküman listesi** alacak. Bu liste içinde sorguya anlamsal olarak en yakın dokümanı bulup döndürecek. Bu, bir önceki alıştırmadaki `search_sbert` fonksiyonuna çok benzer.

In [3]:
def retrieve_best_document(query, department_docs):
    """Verilen bir doküman listesi içinde sorguya en uygun olanı bulur."""
    # TODO: Hem sorguyu hem de departman dokümanlarını Sentence-BERT ile vektörlere dönüştürün.
    embeddings = sbert_model.encode(department_docs + [query])
    
    # TODO: Gerekli embedding'leri ayırın ve kosinüs benzerliğini hesaplayın.
    query_embedding = embeddings[-1]
    doc_embeddings = embeddings[:-1]
    similarities =  cosine_similarity([query_embedding], doc_embeddings)[0]
    
    # En uygun dokümanın indeksini ve skorunu bulun
    best_doc_index = np.argmax(similarities)
    best_score = similarities[best_doc_index]
    
    return department_docs[best_doc_index], best_score

## Adım 4: Uçtan Uca Sistem ve Deney Kurgusu

Şimdi Sınıflandırıcı (Router) ve Bilgi Getirici (Retriever) modüllerini birleştirerek tam bir sorgu yanıtlama sistemi oluşturalım ve test edelim.

In [4]:
def answer_corporate_question(query):
    print("="*70)
    print(f"KULLANICI SORGUSU: '{query}'")
    print("-"*70)
    
    # Adım 1: Sorgunun hangi kategoriye ait olduğunu tahmin et (Yönlendirme)
    query_embedding = sbert_model.encode([query])
    predicted_department = document_classifier.predict(query_embedding)[0]
    print(f"1. Adım [Yönlendirme]: Sorgunun '{predicted_department}' departmanı ile ilgili olduğu tespit edildi.")
    
    # Adım 2: Sadece o kategori içindeki dokümanları al
    relevant_docs = corpus[predicted_department]
    print(f"2. Adım [Filtreleme]: Arama, {len(relevant_docs)} adet doküman içinde yapılacak.")
    
    # Adım 3: Filtrelenmiş dokümanlar içinde en uygun olanı bul (Bilgi Getirme)
    best_answer, score = retrieve_best_document(query, relevant_docs)
    print(f"3. Adım [Bilgi Getirme]: En uygun cevap bulundu (Benzerlik Skoru: {score:.4f}).")
    
    # Sonuç
    print("\n>> SİSTEMİN ÖNERDİĞİ CEVAP:")
    print(f">> {best_answer}")
    print("="*70 + "\n")

# Test edilecek sorgular
test_queries = [
    "Maaşımı ne zaman alacağım?",
    "Yeni bir dizüstü bilgisayarı nasıl talep edebilirim?",
    "VPN'e bağlanamıyorum",
    "Servis saatleri nedir?",
    "Teslimat için harcama raporunu nasıl alabilirim?"
]

# Tüm sorguları çalıştır
for q in test_queries:
    answer_corporate_question(q)

KULLANICI SORGUSU: 'Maaşımı ne zaman alacağım?'
----------------------------------------------------------------------


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

1. Adım [Yönlendirme]: Sorgunun 'Muhasebe' departmanı ile ilgili olduğu tespit edildi.
2. Adım [Filtreleme]: Arama, 4 adet doküman içinde yapılacak.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

3. Adım [Bilgi Getirme]: En uygun cevap bulundu (Benzerlik Skoru: 0.8313).

>> SİSTEMİN ÖNERDİĞİ CEVAP:
>> Maaş ödemeleri her ayın son iş günü banka hesaplarına yatırılır.

KULLANICI SORGUSU: 'Yeni bir dizüstü bilgisayarı nasıl talep edebilirim?'
----------------------------------------------------------------------


...

3. Adım [Bilgi Getirme]: En uygun cevap bulundu (Benzerlik Skoru: 0.8044).

>> SİSTEMİN ÖNERDİĞİ CEVAP:
>> Personel servis güzergahları ve saatleri intranet sayfasında yayınlanmaktadır.

KULLANICI SORGUSU: 'Teslimat için harcama raporunu nasıl alabilirim?'
----------------------------------------------------------------------


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

1. Adım [Yönlendirme]: Sorgunun 'Muhasebe' departmanı ile ilgili olduğu tespit edildi.
2. Adım [Filtreleme]: Arama, 4 adet doküman içinde yapılacak.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

3. Adım [Bilgi Getirme]: En uygun cevap bulundu (Benzerlik Skoru: 0.8467).

>> SİSTEMİN ÖNERDİĞİ CEVAP:
>> Avans talepleri için Muhasebe departmanından onay alınması gerekmektedir.

